# Sudoku RL local notebook

This notebook loads `Qwen/Qwen3-0.6B` from Hugging Face with `transformers` and runs the same Sudoku episode loop as `inference.py`, but fully local for model generation.

If you already have the dependencies installed, you can skip the install cell.

## Kernel check

If `pip install torch` succeeded in a terminal but `import torch` fails here, the notebook is almost certainly using a different Python environment.

Run the next cell first, then install packages with `%pip` inside this notebook so they go into the active kernel.

In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Working directory:", Path.cwd())

try:
    import torch
    print("torch already installed:", torch.__version__)
except ModuleNotFoundError:
    print("torch is not installed in this notebook kernel.")
    print("Run the next install cell with %pip, then restart the kernel.")

In [ ]:
%pip install -q torch transformers accelerate openenv-core[core]

In [ ]:
import asyncio
import json
import os
import re
import textwrap
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from sudoku_rl import SudokuRlAction, SudokuRlEnv

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3-0.6B")
IMAGE_NAME = os.getenv("IMAGE_NAME", "sudoku_rl:latest")
BASE_URL = os.getenv("SUDOKU_BASE_URL", "")
EMPTY_BOXES = int(os.getenv("EMPTY_BOXES", "35"))
MAX_STEPS = int(os.getenv("MAX_STEPS", "60"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "96"))
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.2"))
TOP_P = float(os.getenv("TOP_P", "0.8"))

SYSTEM_PROMPT = textwrap.dedent(
    """
    You are solving a Sudoku puzzle one move at a time.

    You may reason briefly before answering.
    Your final answer must contain exactly one JSON object with this schema:
    {"row": <1-9>, "column": <1-9>, "value": <1-9>}

    Rules:
    - Choose one editable cell and one value from 1 to 9.
    - Prefer correcting currently invalid cells before filling untouched cells.
    - Do not choose cells from the original fixed puzzle.
    - Do not include markdown or code fences.
    - If you include reasoning, keep the final JSON as the last structured object.
    """
).strip()

if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {
        "torch_dtype": torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        "device_map": "auto",
    }
elif torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"torch_dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"torch_dtype": torch.float32}

print(f"Loading {MODEL_NAME} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True, **model_kwargs)
if "device_map" not in model_kwargs:
    model = model.to(device)
model.eval()
print("Model loaded.")

In [ ]:
def build_user_prompt(step: int, observation: Any, history: list[str]) -> str:
    history_block = "\n".join(history[-6:]) if history else "None"
    invalid_block = observation.invalid_cells if observation.invalid_cells else "[]"
    return textwrap.dedent(
        f"""
        Step: {step}
        Status: {observation.status}
        Score: {observation.score}
        Score delta from last move: {observation.score_delta}
        Moves attempted: {observation.moves}
        Mistakes: {observation.mistakes}
        Empty cells remaining: {observation.empty_cells_remaining}
        Incorrect cells: {observation.incorrect_cells}
        Invalid cells (0-based [row, col]): {invalid_block}
        Last environment message: {observation.message}

        Current board:
        {observation.board_text}

        Previous recent actions and responses:
        {history_block}

        Return the next move as JSON.
        """
    ).strip()


def _extract_digit_1_to_9(value: Any) -> int | None:
    if isinstance(value, int):
        return value if 1 <= value <= 9 else None

    if isinstance(value, str):
        match = re.search(r"\b([1-9])\b", value)
        if match:
            return int(match.group(1))

    return None


def parse_action(text: str) -> SudokuRlAction | None:
    cleaned = (text or "").strip()
    if not cleaned:
        return None

    if "</think>" in cleaned:
        cleaned = cleaned.split("</think>", 1)[1].strip()

    candidates: list[dict[str, Any]] = []
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            candidates.append(parsed)
    except json.JSONDecodeError:
        pass

    for match in re.finditer(r"\{[^{}]+\}", cleaned, flags=re.DOTALL):
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            candidates.append(parsed)

    row_match = re.search(r'"?row"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    column_match = re.search(r'"?(?:column|col)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    value_match = re.search(r'"?(?:value|number)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    if row_match and column_match and value_match:
        candidates.append(
            {
                "row": int(row_match.group(1)),
                "column": int(column_match.group(1)),
                "value": int(value_match.group(1)),
            }
        )

    for candidate in candidates:
        try:
            row = _extract_digit_1_to_9(candidate["row"])
            column = _extract_digit_1_to_9(candidate["column"])
            value = _extract_digit_1_to_9(candidate.get("value", candidate.get("number")))
        except (KeyError, TypeError, ValueError):
            continue
        if row is not None and column is not None and value is not None:
            return SudokuRlAction(row=row, column=column, value=value)

    return None


def candidates_for_cell(board: list[list[int]], row_index: int, column_index: int) -> list[int]:
    if board[row_index][column_index] != 0:
        return []

    used_values = set(board[row_index])
    used_values.update(board[r][column_index] for r in range(9))

    box_row = (row_index // 3) * 3
    box_column = (column_index // 3) * 3
    for sub_row in range(box_row, box_row + 3):
        for sub_column in range(box_column, box_column + 3):
            used_values.add(board[sub_row][sub_column])

    return [value for value in range(1, 10) if value not in used_values]


def heuristic_action(observation: Any) -> SudokuRlAction:
    board = [row[:] for row in observation.board]
    invalid_targets = {
        (int(row_index), int(column_index))
        for row_index, column_index in observation.invalid_cells
        if 0 <= row_index < 9 and 0 <= column_index < 9
    }

    for row_index, column_index in invalid_targets:
        board[row_index][column_index] = 0

    target_cells: list[tuple[int, int]] = list(invalid_targets)
    for row_index in range(9):
        for column_index in range(9):
            if observation.initial_puzzle[row_index][column_index] != 0:
                continue
            if board[row_index][column_index] == 0 and (row_index, column_index) not in invalid_targets:
                target_cells.append((row_index, column_index))

    if not target_cells:
        for row_index in range(9):
            for column_index in range(9):
                if observation.initial_puzzle[row_index][column_index] == 0:
                    target_cells.append((row_index, column_index))

    for row_index, column_index in target_cells:
        values = candidates_for_cell(board, row_index, column_index)
        if values:
            return SudokuRlAction(row=row_index + 1, column=column_index + 1, value=values[0])

    row_index, column_index = target_cells[0] if target_cells else (0, 0)
    return SudokuRlAction(row=row_index + 1, column=column_index + 1, value=1)

In [ ]:
def generate_model_text(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        try:
            model_inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        except TypeError:
            model_inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
    else:
        prompt_text = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
        model_inputs = tokenizer(prompt_text, return_tensors="pt")
    if "device_map" in model_kwargs:
        target_device = next(model.parameters()).device
        model_inputs = {key: value.to(target_device) for key, value in model_inputs.items()}
    else:
        model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

    generation_kwargs = {
        **model_inputs,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
        "temperature": TEMPERATURE if TEMPERATURE > 0 else None,
        "top_p": TOP_P if TEMPERATURE > 0 else None,
        "pad_token_id": tokenizer.eos_token_id,
    }
    generation_kwargs = {k: v for k, v in generation_kwargs.items() if v is not None}

    with torch.no_grad():
        outputs = model.generate(**generation_kwargs)

    generated_ids = outputs[0][model_inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    if think_match:
        print("[MODEL_REASONING]")
        print(think_match.group(1).strip())
    print("[MODEL_OUTPUT]")
    print(text)
    return text


def get_local_action(step: int, observation: Any, history: list[str]) -> tuple[SudokuRlAction, str, str | None, str]:
    user_prompt = build_user_prompt(step, observation, history)
    try:
        text = generate_model_text(user_prompt)
        action = parse_action(text)
        if action is not None:
            return action, "model", None, text
        return heuristic_action(observation), "heuristic_fallback", f"unparseable_model_output={text!r}", text
    except Exception as exc:
        return heuristic_action(observation), "heuristic_fallback", str(exc), ""

In [ ]:
async def create_env():
    if BASE_URL:
        return SudokuRlEnv(base_url=BASE_URL)
    return await SudokuRlEnv.from_docker_image(IMAGE_NAME)


async def run_episode(empty_boxes: int = EMPTY_BOXES, max_steps: int = MAX_STEPS):
    env = await create_env()
    history: list[str] = []
    rewards: list[float] = []
    final_status = "unknown"
    final_score_raw = 0

    try:
        result = await env.reset(empty_boxes=empty_boxes)
        observation = result.observation
        print("[START]", f"empty_boxes={empty_boxes}", f"model={MODEL_NAME}")
        print(observation.board_text)

        for step in range(1, max_steps + 1):
            if result.done:
                break

            action, source, error, raw_text = get_local_action(step, observation, history)
            result = await env.step(action)
            observation = result.observation

            reward = float(result.reward or 0.0)
            rewards.append(reward)
            final_status = observation.status
            final_score_raw = observation.score

            print(
                "[STEP]",
                f"step={step}",
                f"row={action.row}",
                f"column={action.column}",
                f"value={action.value}",
                f"reward={reward:.2f}",
                f"score={observation.score}",
                f"status={observation.status}",
                f"source={source}",
                f"error={error or 'null'}",
            )
            print(observation.message)

            history.append(
                f"step={step} raw={raw_text!r} action={{row:{action.row}, column:{action.column}, value:{action.value}}} "
                f"reward={reward:+.2f} status={observation.status} message={observation.message}"
            )

            if result.done:
                break

        max_possible_score = max(observation.empty_boxes * max(observation.score_step, 1), 1)
        normalized_score = max(min(final_score_raw / max_possible_score, 1.0), 0.0)
        success = final_status == "solved"
        print("[END]", f"success={success}", f"score={normalized_score:.3f}", f"raw_score={final_score_raw}")
        return {
            "success": success,
            "normalized_score": normalized_score,
            "raw_score": final_score_raw,
            "final_status": final_status,
            "rewards": rewards,
            "history": history,
            "board_text": observation.board_text,
        }
    finally:
        await env.close()

In [ ]:
result = await run_episode()
result